In [217]:
import pandas as pd
from PIL import Image
from tqdm import tqdm
import numpy as np
import cv2

from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as v2
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchmetrics as tm

In [218]:
root_path = "/home/stefan/ioai-prep/kits/sami"
device = "cuda" if torch.cuda.is_available() else "cpu"

seed = 42
torch.random.manual_seed(seed)

# Data

In [219]:
train_df = pd.read_csv(f"{root_path}/train_data.csv")
train_df.head()

,image_path,rotation_label
0,q_d12f4bab49c943d09ae1d9bc30bed8c8.jpg,2
1,q_d5d38b9096674d7c873ee3e72075e7dc.jpg,3
2,q_f395bac9b9b6432f8dd07e62d90a9a5f.jpg,2
3,q_ed5eb4c009b54ff98e3d82de95a87c2a.jpg,3
4,q_382300a929724b08bda84a2856841c23.jpg,0


In [ ]:
class ImgDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        super().__init__()

        self.df = df

        self.transforms = v2.Compose([
            v2.Resize(224),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True)
        ])

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = f'{root_path}/images/{row["image_path"]}'
        img = Image.open(img_path)

        if "rotation_label" in self.df:
            img = self.transforms(img)
            label = torch.tensor(row["rotation_label"], dtype=torch.long)
            return img, label

        if "gallery_id" in self.df:
            img = self.transforms(img)
            id = row["gallery_id"]
            return img, id

        return self.transforms(img)

    def __len__(self):
        return len(self.df)

In [221]:
train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=seed)

train_ds = ImgDataset(train_df)
val_ds = ImgDataset(val_df)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)

In [222]:
batch = next(iter(train_loader))
[b.shape for b in batch]

[torch.Size([32, 3, 224, 224]), torch.Size([32])]

# Model

In [223]:
class TransformDetModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = models.resnet18(weights=None)
        self.encoder.fc = nn.Identity()

        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(512, 4)
        )

    def forward(self, x):
        embd = self.encoder(x)
        return self.head(embd)

In [234]:
model = TransformDetModel().to(device)

model(batch[0].to(device)).shape

torch.Size([32, 4])

# Subtask 1

## Training

In [235]:
epochs = 40
lr = 1e-3
best_metric = -float('inf')

optimizer = torch.optim.AdamW(model.parameters(), lr, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
f1 = tm.F1Score(task='multiclass', num_classes=4, average='macro').to(device)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

In [236]:
for epoch in range(1, epochs+1):
    model.train()
    running_loss = 0.0

    for img, label in tqdm(train_loader):
        img, label = img.to(device), label.to(device)

        out = model(img)
        loss = criterion(out, label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    model.eval()
    running_val_loss = 0.0
    f1.reset()

    for img, label in tqdm(val_loader):
        img, label = img.to(device), label.to(device)

        with torch.no_grad():
            out = model(img)
            loss = criterion(out, label)
            f1(out, label)

        running_val_loss += loss.item()

    avg_train_loss = running_loss / len(train_loader)
    avg_val_loss = running_val_loss / len(val_loader)
    f1_metric = f1.compute()

    if f1_metric > best_metric:
        best_metric = f1_metric
        print("new best!")
        torch.save(model.state_dict(), "best.pth")
    scheduler.step(f1_metric)

    print(f"epoch {epoch}: train_loss={avg_train_loss:.3f}, val_loss={avg_val_loss:.3f}, f1={f1_metric:.3f}")

  0%|          | 0/100 [00:00<?, ?it/s]

100%|██████████| 25/25 [00:01<00:00, 24.00it/s]


new best!
epoch 1: train_loss=1.446, val_loss=2.707, f1=0.231


100%|██████████| 25/25 [00:01<00:00, 24.42it/s]


new best!
epoch 2: train_loss=1.339, val_loss=1.279, f1=0.345


...

new best!
epoch 38: train_loss=0.019, val_loss=1.852, f1=0.625


100%|██████████| 25/25 [00:01<00:00, 21.82it/s]


epoch 39: train_loss=0.007, val_loss=1.830, f1=0.623


100%|██████████| 25/25 [00:01<00:00, 22.38it/s]


new best!
epoch 40: train_loss=0.005, val_loss=1.862, f1=0.633


In [237]:
model.load_state_dict(torch.load("best.pth"))
model.eval()
print(best_metric)

tensor(0.6332, device='cuda:0')


## Inference

In [239]:
test_df = pd.read_csv(f"{root_path}/test_data.csv")
test_ds = ImgDataset(test_df)
test_loader = DataLoader(test_ds, batch_size=32)

test_df.head()

,datapoint_id,image_path
0,10000,q_44f7218929e04ca7bd4e2d0be808ebbe.jpg
1,10001,q_2ace1caa6d954fc4849efe43cf12584d.jpg
2,10002,q_e687e723a4cd412b8779fd5903c3e082.jpg
3,10003,q_14e58cc7f93946f0a1404611288a8078.jpg
4,10004,q_52a65e6ce2be48199b6d2de2e700583e.jpg


In [240]:
subtask1 = []

for img in tqdm(test_loader):
    img = img.to(device)

    pred = model(img).to(device)
    pred = torch.argmax(pred, dim=1)
    subtask1.extend(pred.cpu().tolist())

  0%|          | 0/32 [00:00<?, ?it/s]

100%|██████████| 32/32 [00:01<00:00, 17.16it/s]


# Subtask 2

In [ ]:
vision_df = pd.read_csv(f"{root_path}/vision_catalog.csv")
gallery_ids = vision_df["gallery_id"].values

subtask2 = []

In [252]:
sift = cv2.SIFT_create(nfeatures=2000, contrastThreshold=0.02, edgeThreshold=15)
clahe = cv2.createCLAHE(clipLimit=2.5)
flann = cv2.FlannBasedMatcher(dict(algorithm=1, trees=5), dict(checks=50))
bf = cv2.BFMatcher()

In [253]:
all_descriptors = []
descriptor_to_id = []
catalog_des = []
catalog_kp = []
catalog_ids = vision_df["gallery_id"].values

for i, row in enumerate(vision_df.itertuples()):
    img = cv2.imread(f"{root_path}/images/{row.image_path}", 0)
    img = clahe.apply(img)

    kp, des = sift.detectAndCompute(img, None)

    if des is not None:
        des /= des.sum(axis=1, keepdims=True) + 1e-7
        des = np.sqrt(des)

        all_descriptors.append(des)
        descriptor_to_id.extend([i] * len(des))
        catalog_des.append(des)
        catalog_kp.append(np.array([k.pt for k in kp]))
    else:
        catalog_des.append(np.array([]))
        catalog_kp.append(np.array([]))

all_descriptors = np.vstack(all_descriptors).astype(np.float32)
descriptor_to_id = np.array(descriptor_to_id)
flann.add([all_descriptors])
flann.train()

In [ ]:
for i, row in tqdm(test_df.iterrows(), total=len(test_df)):
    img = cv2.imread(f"{root_path}/images/{row.image_path}", 0)

    img = clahe.apply(img)
    kp_q, des_q = sift.detectAndCompute(img, None)

    if des_q is None:
        subtask2.append(catalog_ids[0])
        continue

    des_q /= des_q.sum(axis=1, keepdims=True) + 1e-7
    des_q = np.sqrt(des_q)

    matches = flann.knnMatch(des_q, k=2)
    votes = np.zeros(len(catalog_ids))
    for m, n in matches:
        if m.distance < 0.8 * n.distance:
            votes[descriptor_to_id[m.trainIdx]] += 1

    top_k = np.argsort(votes)[-5:][::-1]
    best_idx = top_k[0]
    best_inliers = 0

    for cand_idx in top_k:
        des_c = catalog_des[cand_idx]
        pts_c = catalog_kp[cand_idx]

        if len(des_c) < 4:
            continue

        m_list = bf.knnMatch(des_q, des_c, k=2)
        good = [m for m, n in m_list if m.distance < 0.75 * n.distance]

        if len(good) >= 4:
            src = np.float32([kp_q[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
            dst = np.float32([pts_c[m.trainIdx] for m in good]).reshape(-1, 1, 2)

            _, mask = cv2.findHomography(src, dst, cv2.RANSAC, 5.0)
            inliers = np.sum(mask) if mask is not None else 0

            if inliers > best_inliers:
                best_inliers = inliers
                best_idx = cand_idx

    subtask2.append(catalog_ids[best_idx])

100%|██████████| 1000/1000 [00:46<00:00, 21.39it/s]


# Submission

In [255]:
def build_subtask(sid, answers):
    return pd.DataFrame({
        "subtaskID": sid,
        "datapointID": test_df["datapoint_id"],
        "answer": answers
    })

subtasks = [
    (1, subtask1),
    (2, subtask2)
]

submission = pd.concat([build_subtask(sid, ans) for sid, ans in subtasks], axis=0)

In [256]:
submission.head()

,subtaskID,datapointID,answer
0,1,10000,3
1,1,10001,1
2,1,10002,1
3,1,10003,3
4,1,10004,2


In [257]:
submission.to_csv(f"{root_path}/submission.csv", index=False)